In [1]:
from pathlib import Path
import ast
import json

import numpy as np
import pandas as pd
import wfdb

from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, resample_poly

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = Path("../../").resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "ecg" / "ptbxl"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ecg" / "ptbxl"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw PTB-XL:", RAW_DIR)
print("Processed PTB-XL:", PROCESSED_DIR)

Raw PTB-XL: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\raw\ecg\ptbxl
Processed PTB-XL: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl


In [3]:
DATABASE_FILE = RAW_DIR / "ptbxl_database.csv"
SCP_FILE = RAW_DIR / "scp_statements.csv"

print("ptbxl_database.csv:", DATABASE_FILE.exists())
print("scp_statements.csv:", SCP_FILE.exists())
print("records100:", (RAW_DIR / "records100").exists())

if not DATABASE_FILE.exists():
    raise FileNotFoundError(DATABASE_FILE)

if not SCP_FILE.exists():
    raise FileNotFoundError(SCP_FILE)

ptbxl_database.csv: True
scp_statements.csv: True
records100: True


In [4]:
ptbxl = pd.read_csv(
    DATABASE_FILE,
    index_col=0
)

scp = pd.read_csv(
    SCP_FILE,
    index_col=0
)

print("PTB-XL shape:", ptbxl.shape)
print("SCP statements shape:", scp.shape)

display(ptbxl.head())

PTB-XL shape: (21837, 27)
SCP statements shape: (71, 12)


,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr


In [5]:
print("PTB-XL columns:")
print(ptbxl.columns.tolist())

print("\nSCP statement columns:")
print(scp.columns.tolist())

PTB-XL columns:
['patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion', 'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']

SCP statement columns:
['description', 'diagnostic', 'form', 'rhythm', 'diagnostic_class', 'diagnostic_subclass', 'Statement Category', 'SCP-ECG Statement Description', 'AHA code', 'aECG REFID', 'CDISC Code', 'DICOM Code']


In [6]:
def parse_scp_codes(value):
    if pd.isna(value):
        return {}

    if isinstance(value, dict):
        return value

    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return {}


ptbxl["scp_codes_parsed"] = ptbxl["scp_codes"].apply(
    parse_scp_codes
)

print(
    "Example:",
    ptbxl["scp_codes_parsed"].iloc[0]
)

Example: {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}


In [7]:
from sklearn.model_selection import train_test_split

patients = (
    ptbxl["patient_id"]
    .dropna()
    .unique()
)

train_patients, temp_patients = train_test_split(
    patients,
    test_size=0.30,
    random_state=42
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=42
)

train_patients = set(train_patients)
val_patients = set(val_patients)
test_patients = set(test_patients)

ptbxl["split"] = "unknown"

ptbxl.loc[
    ptbxl["patient_id"].isin(train_patients),
    "split"
] = "train"

ptbxl.loc[
    ptbxl["patient_id"].isin(val_patients),
    "split"
] = "val"

ptbxl.loc[
    ptbxl["patient_id"].isin(test_patients),
    "split"
] = "test"

print(ptbxl["split"].value_counts())

split
train    15298
test      3273
val       3266
Name: count, dtype: int64


In [8]:
print("Available SCP statement columns:")
print(scp.columns.tolist())

diagnostic_column = None

for candidate in ["diagnostic", "diagnostic_class"]:
    if candidate in scp.columns:
        diagnostic_column = candidate
        break

print("\nDiagnostic column:", diagnostic_column)

Available SCP statement columns:
['description', 'diagnostic', 'form', 'rhythm', 'diagnostic_class', 'diagnostic_subclass', 'Statement Category', 'SCP-ECG Statement Description', 'AHA code', 'aECG REFID', 'CDISC Code', 'DICOM Code']

Diagnostic column: diagnostic


In [9]:
if diagnostic_column is not None:
    print(
        scp[diagnostic_column]
        .dropna()
        .value_counts()
    )

    display(
        scp[[diagnostic_column]]
        .dropna()
        .drop_duplicates()
        .head(20)
    )

diagnostic
1.0    44
Name: count, dtype: int64


,diagnostic
NDT,1.0


In [10]:
DIAGNOSTIC_LABELS = [
    "NORM",
    "MI",
    "STTC",
    "CD",
    "HYP",
]

diagnostic_code_to_class = {}

if (
    "diagnostic" in scp.columns
    and "diagnostic_class" in scp.columns
):
    for code, row in scp.iterrows():

        if pd.isna(row["diagnostic"]):
            continue

        if row["diagnostic"] == 1:
            diagnostic_class = row["diagnostic_class"]

            if diagnostic_class in DIAGNOSTIC_LABELS:
                diagnostic_code_to_class[code] = (
                    diagnostic_class
                )

print(
    "Diagnostic mappings:",
    len(diagnostic_code_to_class)
)

print(
    list(diagnostic_code_to_class.items())[:20]
)

Diagnostic mappings: 44
[('NDT', 'STTC'), ('NST_', 'STTC'), ('DIG', 'STTC'), ('LNGQT', 'STTC'), ('NORM', 'NORM'), ('IMI', 'MI'), ('ASMI', 'MI'), ('LVH', 'HYP'), ('LAFB', 'CD'), ('ISC_', 'STTC'), ('IRBBB', 'CD'), ('1AVB', 'CD'), ('IVCD', 'CD'), ('ISCAL', 'STTC'), ('CRBBB', 'CD'), ('CLBBB', 'CD'), ('ILMI', 'MI'), ('LAO/LAE', 'HYP'), ('AMI', 'MI'), ('ALMI', 'MI')]


In [11]:
def get_diagnostic_targets(codes):

    targets = {
        label: 0
        for label in DIAGNOSTIC_LABELS
    }

    for code in codes.keys():

        label = diagnostic_code_to_class.get(code)

        if label is not None:
            targets[label] = 1

    return targets


diagnostic_targets = ptbxl[
    "scp_codes_parsed"
].apply(get_diagnostic_targets)

diagnostic_df = pd.DataFrame(
    diagnostic_targets.tolist(),
    index=ptbxl.index
)

for label in DIAGNOSTIC_LABELS:
    ptbxl[label] = diagnostic_df[label].astype(np.float32)

display(
    ptbxl[
        ["scp_codes_parsed"] + DIAGNOSTIC_LABELS
    ].head()
)

,scp_codes_parsed,NORM,MI,STTC,CD,HYP
ecg_id,,,,,,
1,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}",1.0,0.0,0.0,0.0,0.0
2,"{'NORM': 80.0, 'SBRAD': 0.0}",1.0,0.0,0.0,0.0,0.0
3,"{'NORM': 100.0, 'SR': 0.0}",1.0,0.0,0.0,0.0,0.0
4,"{'NORM': 100.0, 'SR': 0.0}",1.0,0.0,0.0,0.0,0.0
5,"{'NORM': 100.0, 'SR': 0.0}",1.0,0.0,0.0,0.0,0.0


In [12]:
RHYTHM_LABELS = [
    "SR",
    "AFIB",
    "AFLT",
    "STACH",
    "SBRAD",
    "SARRH",
    "PSVT",
    "BIGU",
    "PACE",
]

rhythm_code_set = set()

if "rhythm" in scp.columns:

    for code, row in scp.iterrows():

        if row["rhythm"] == 1:
            rhythm_code_set.add(code)

print("Number of rhythm SCP codes:", len(rhythm_code_set))
print("Example codes:", list(rhythm_code_set)[:20])

Number of rhythm SCP codes: 12
Example codes: ['SARRH', 'SVTAC', 'TRIGU', 'AFIB', 'AFLT', 'PSVT', 'SBRAD', 'STACH', 'SVARR', 'PACE', 'BIGU', 'SR']


In [13]:
def get_rhythm_targets(codes):

    targets = {
        label: 0
        for label in RHYTHM_LABELS
    }

    for code in codes.keys():

        if code in RHYTHM_LABELS:
            targets[code] = 1

    return targets


rhythm_targets = ptbxl[
    "scp_codes_parsed"
].apply(get_rhythm_targets)

rhythm_df = pd.DataFrame(
    rhythm_targets.tolist(),
    index=ptbxl.index
)

for label in RHYTHM_LABELS:
    ptbxl[
        f"RHYTHM_{label}"
    ] = rhythm_df[label].astype(np.float32)

display(
    ptbxl[
        [f"RHYTHM_{x}" for x in RHYTHM_LABELS]
    ].head()
)

,RHYTHM_SR,RHYTHM_AFIB,RHYTHM_AFLT,RHYTHM_STACH,RHYTHM_SBRAD,RHYTHM_SARRH,RHYTHM_PSVT,RHYTHM_BIGU,RHYTHM_PACE
ecg_id,,,,,,,,,
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
TARGET_FS = 100
DURATION_SECONDS = 10

NUM_LEADS = 12
TARGET_LENGTH = TARGET_FS * DURATION_SECONDS

LEAD_NAMES = [
    "I",
    "II",
    "III",
    "aVR",
    "aVL",
    "aVF",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
]

print("Sampling rate:", TARGET_FS)
print("Duration:", DURATION_SECONDS)
print("Target length:", TARGET_LENGTH)
print("Leads:", NUM_LEADS)

Sampling rate: 100
Duration: 10
Target length: 1000
Leads: 12


In [15]:
def highpass_filter(
    signal,
    fs,
    cutoff=0.5,
    order=3
):
    sos = butter(
        order,
        cutoff,
        btype="highpass",
        fs=fs,
        output="sos"
    )

    return sosfiltfilt(
        sos,
        signal,
        axis=0
    )

In [16]:
def resample_signal(
    signal,
    original_fs,
    target_fs=100
):
    if int(original_fs) == int(target_fs):
        return signal

    gcd = np.gcd(
        int(original_fs),
        int(target_fs)
    )

    up = int(target_fs) // gcd
    down = int(original_fs) // gcd

    return resample_poly(
        signal,
        up,
        down,
        axis=0
    )

In [17]:
def fix_signal_length(
    signal,
    target_length=1000
):
    current_length = signal.shape[0]

    if current_length > target_length:
        return signal[:target_length]

    if current_length < target_length:
        padding = np.zeros(
            (
                target_length - current_length,
                signal.shape[1]
            ),
            dtype=signal.dtype
        )

        return np.vstack(
            [signal, padding]
        )

    return signal

In [18]:
def normalize_signal(signal):

    mean = signal.mean(
        axis=0,
        keepdims=True
    )

    std = signal.std(
        axis=0,
        keepdims=True
    )

    std = np.where(
        std < 1e-8,
        1.0,
        std
    )

    return (
        (signal - mean) / std
    ).astype(np.float32)

In [19]:
filename_column = None

for candidate in [
    "filename_lr",
    "filename_hr"
]:
    if candidate in ptbxl.columns:
        filename_column = candidate
        break

if filename_column is None:
    raise KeyError(
        "No PTB-XL waveform filename column found."
    )

sample_record = ptbxl.iloc[0][
    filename_column
]

sample_path = RAW_DIR / sample_record

print("Filename column:", filename_column)
print("Sample record:", sample_record)
print("Path:", sample_path)

Filename column: filename_lr
Sample record: records100/00000/00001_lr
Path: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\raw\ecg\ptbxl\records100\00000\00001_lr


In [20]:
signal, fields = wfdb.rdsamp(
    str(sample_path)
)

original_fs = fields["fs"]

print("Original shape:", signal.shape)
print("Original sampling rate:", original_fs)

signal = highpass_filter(
    signal,
    fs=original_fs
)

signal = resample_signal(
    signal,
    original_fs=original_fs,
    target_fs=TARGET_FS
)

signal = fix_signal_length(
    signal,
    target_length=TARGET_LENGTH
)

signal = normalize_signal(signal)

print("Processed shape:", signal.shape)
print("Processed dtype:", signal.dtype)

Original shape: (1000, 12)
Original sampling rate: 100
Processed shape: (1000, 12)
Processed dtype: float32


In [21]:
WAVEFORM_DIR = PROCESSED_DIR / "waveforms"

WAVEFORM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Waveform output:", WAVEFORM_DIR)

Waveform output: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\waveforms


In [22]:
MAX_RECORDS = None
# Set to e.g. 100 for a quick test.
# Set to None to process all available records.

records_to_process = ptbxl

if MAX_RECORDS is not None:
    records_to_process = ptbxl.iloc[:MAX_RECORDS]

processed_rows = []

for i, (index, row) in enumerate(
    records_to_process.iterrows()
):

    record_name = row[filename_column]

    source_path = (
        RAW_DIR / record_name
    )

    try:

        signal, fields = wfdb.rdsamp(
            str(source_path)
        )

        original_fs = fields["fs"]

        signal = highpass_filter(
            signal,
            fs=original_fs
        )

        signal = resample_signal(
            signal,
            original_fs=original_fs,
            target_fs=TARGET_FS
        )

        signal = fix_signal_length(
            signal,
            target_length=TARGET_LENGTH
        )

        signal = normalize_signal(signal)

        output_name = (
            Path(record_name).with_suffix(".npy").name
        )

        output_path = (
            WAVEFORM_DIR / output_name
        )

        np.save(
            output_path,
            signal
        )

        processed_rows.append({
            "record_id": index,
            "patient_id": row["patient_id"],
            "record_name": record_name,
            "processed_path": str(
                output_path.relative_to(PROJECT_ROOT)
            ),
            "split": row["split"],
            "original_fs": original_fs,
        })

    except Exception as e:

        print(
            f"Skipped {record_name}: {e}"
        )

    if (i + 1) % 500 == 0:
        print(
            f"Processed {i + 1} records"
        )

Processed 500 records
Processed 1000 records
Processed 1500 records
Processed 2000 records
Processed 2500 records
Processed 3000 records
Processed 3500 records
Processed 4000 records
Processed 4500 records
Processed 5000 records
Processed 5500 records
Processed 6000 records
Processed 6500 records
Processed 7000 records
Processed 7500 records
Processed 8000 records
Processed 8500 records
Processed 9000 records
Processed 9500 records
Processed 10000 records
Processed 10500 records
Processed 11000 records
Processed 11500 records
Processed 12000 records
Processed 12500 records
Processed 13000 records
Processed 13500 records
Processed 14000 records
Processed 14500 records
Processed 15000 records
Processed 15500 records
Processed 16000 records
Processed 16500 records
Processed 17000 records
Processed 17500 records
Processed 18000 records
Processed 18500 records
Processed 19000 records
Processed 19500 records
Processed 20000 records
Processed 20500 records
Processed 21000 records
Processed 21

In [23]:
processed_manifest = pd.DataFrame(
    processed_rows
)

print(
    "Processed records:",
    len(processed_manifest)
)

display(
    processed_manifest.head()
)

Processed records: 21837


,record_id,patient_id,record_name,processed_path,split,original_fs
0,1,15709.0,records100/00000/00001_lr,data\processed\ecg\ptbxl\waveforms\00001_lr.npy,test,100
1,2,13243.0,records100/00000/00002_lr,data\processed\ecg\ptbxl\waveforms\00002_lr.npy,train,100
2,3,20372.0,records100/00000/00003_lr,data\processed\ecg\ptbxl\waveforms\00003_lr.npy,train,100
3,4,17014.0,records100/00000/00004_lr,data\processed\ecg\ptbxl\waveforms\00004_lr.npy,test,100
4,5,17448.0,records100/00000/00005_lr,data\processed\ecg\ptbxl\waveforms\00005_lr.npy,train,100


In [24]:
label_columns = (
    DIAGNOSTIC_LABELS
    + [f"RHYTHM_{x}" for x in RHYTHM_LABELS]
)

label_lookup = ptbxl[
    ["patient_id", "split"] + label_columns
].copy()

label_lookup["record_name"] = (
    ptbxl[filename_column]
)

processed_manifest = processed_manifest.merge(
    label_lookup,
    on=["patient_id", "split", "record_name"],
    how="left"
)

print(
    "Manifest shape:",
    processed_manifest.shape
)

display(
    processed_manifest.head()
)

Manifest shape: (21837, 20)


,record_id,patient_id,record_name,processed_path,split,original_fs,NORM,MI,STTC,CD,HYP,RHYTHM_SR,RHYTHM_AFIB,RHYTHM_AFLT,RHYTHM_STACH,RHYTHM_SBRAD,RHYTHM_SARRH,RHYTHM_PSVT,RHYTHM_BIGU,RHYTHM_PACE
0,1,15709.0,records100/00000/00001_lr,data\processed\ecg\ptbxl\waveforms\00001_lr.npy,test,100,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,13243.0,records100/00000/00002_lr,data\processed\ecg\ptbxl\waveforms\00002_lr.npy,train,100,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,3,20372.0,records100/00000/00003_lr,data\processed\ecg\ptbxl\waveforms\00003_lr.npy,train,100,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,17014.0,records100/00000/00004_lr,data\processed\ecg\ptbxl\waveforms\00004_lr.npy,test,100,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,17448.0,records100/00000/00005_lr,data\processed\ecg\ptbxl\waveforms\00005_lr.npy,train,100,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
sample_paths = processed_manifest[
    "processed_path"
].head(10)

for relative_path in sample_paths:

    array_path = PROJECT_ROOT / relative_path

    x = np.load(array_path)

    print(
        array_path.name,
        "->",
        x.shape,
        "| NaN:",
        np.isnan(x).sum(),
        "| Inf:",
        np.isinf(x).sum()
    )

00001_lr.npy -> (1000, 12) | NaN: 0 | Inf: 0
00002_lr.npy -> (1000, 12) | NaN: 0 | Inf: 0
00003_lr.npy -> (1000, 12) | NaN: 0 | Inf: 0
00004_lr.npy -> (1000, 12) | NaN: 0 | Inf: 0
00005_lr.npy -> (1000, 12) | NaN: 0 | Inf: 0
00006_lr.npy -> (1000, 12) | NaN: 0 | Inf: 0
00007_lr.npy -> (1000, 12) | NaN: 0 | Inf: 0
00008_lr.npy -> (1000, 12) | NaN: 0 | Inf: 0
00009_lr.npy -> (1000, 12) | NaN: 0 | Inf: 0
00010_lr.npy -> (1000, 12) | NaN: 0 | Inf: 0


In [26]:
manifest_path = (
    PROCESSED_DIR /
    "ptbxl_manifest.csv"
)

processed_manifest.to_csv(
    manifest_path,
    index=False
)

print("Saved:", manifest_path)

Saved: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\ptbxl_manifest.csv


In [27]:
preprocessing_config = {
    "dataset": "PTB-XL",
    "sampling_rate_hz": TARGET_FS,
    "duration_seconds": DURATION_SECONDS,
    "number_of_leads": NUM_LEADS,
    "target_length": TARGET_LENGTH,
    "normalization": "per_record_per_lead_zscore",
    "baseline_wander_filter": {
        "enabled": True,
        "type": "highpass",
        "cutoff_hz": 0.5,
        "order": 3,
    },
    "diagnostic_labels": DIAGNOSTIC_LABELS,
    "rhythm_labels": RHYTHM_LABELS,
    "multilabel": True,
}

config_path = (
    PROCESSED_DIR /
    "preprocessing_config.json"
)

with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        preprocessing_config,
        f,
        indent=2
    )

print("Saved:", config_path)

Saved: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\preprocessing_config.json


In [28]:
print("Diagnostic labels:")

for label in DIAGNOSTIC_LABELS:
    count = int(
        processed_manifest[label].sum()
    )

    print(
        f"{label:6} : "
        f"{count:,}"
    )

print("\nRhythm labels:")

for label in RHYTHM_LABELS:
    column = f"RHYTHM_{label}"

    count = int(
        processed_manifest[column].sum()
    )

    print(
        f"{label:6} : "
        f"{count:,}"
    )

Diagnostic labels:
NORM   : 9,528
MI     : 5,486
STTC   : 5,250
CD     : 4,907
HYP    : 2,655

Rhythm labels:
SR     : 16,782
AFIB   : 1,514
AFLT   : 73
STACH  : 826
SBRAD  : 637
SARRH  : 772
PSVT   : 24
BIGU   : 82
PACE   : 296


In [29]:
print("======================================")
print("PTB-XL preprocessing complete")
print("======================================")

print(
    f"Processed ECGs : "
    f"{len(processed_manifest):,}"
)

print(
    f"Sampling rate  : "
    f"{TARGET_FS} Hz"
)

print(
    f"Input length   : "
    f"{TARGET_LENGTH} samples"
)

print(
    f"Leads          : "
    f"{NUM_LEADS}"
)

print(
    f"Train          : "
    f"{(processed_manifest['split'] == 'train').sum():,}"
)

print(
    f"Validation     : "
    f"{(processed_manifest['split'] == 'val').sum():,}"
)

print(
    f"Test           : "
    f"{(processed_manifest['split'] == 'test').sum():,}"
)

print(
    "\nManifest:",
    manifest_path
)

PTB-XL preprocessing complete
Processed ECGs : 21,837
Sampling rate  : 100 Hz
Input length   : 1000 samples
Leads          : 12
Train          : 15,298
Validation     : 3,266
Test           : 3,273

Manifest: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\ptbxl_manifest.csv
